# `acm.py`: formulas to code

Этот ноутбук нужен не для результатов, а для чтения реализации `NominalACM` в [acm.py](acm.py).

Что здесь есть:
- пошаговый flow метода `__init__`;
- на каждом шаге формула, имена переменных в коде и точный фрагмент `acm.py`;
- акцент на том, что именно считается в реализации.

Что здесь не делаем:
- не обсуждаем презентацию;
- не строим итоговые графики;
- не оцениваем, хороша модель или нет.


In [1]:
from pathlib import Path
from typing import Optional

from IPython.display import Markdown, display

ACM_PATH = Path("acm.py")
ACM_LINES = ACM_PATH.read_text().splitlines()


def show_acm_lines(start: int, end: int, title: Optional[str] = None) -> None:
    body = "\n".join(f"{i:4d}: {ACM_LINES[i - 1]}" for i in range(start, end + 1))
    header = f"**{title}**\n\n" if title else ""
    display(Markdown(header + f"```python\n{body}\n```"))


## Обозначения

Используем стандартные обозначения ACM и сразу связываем их с именами в коде.

| Математика | Смысл | Имя в `acm.py` |
|---|---|---|
| $y_t^{(n)}$ | доходность нулевого купона для срока $n$ месяцев | `curve`, `curve_monthly` |
| $p_t^{(n)} = - \frac{n}{12} y_t^{(n)}$ | лог-цена облигации | `log_prices` |
| $r_t^{(1)}$ | короткая ставка на один период | `short_rate_proxy` |
| $rx_{t+1}^{(n)}$ | excess return | `rx_m` |
| $X_t$ | вектор факторного состояния | `pc_factors_m` |
| $v_{t+1}$ | инновации VAR | `v` |
| $(\mu, \Phi, \Sigma)$ | параметры VAR(1) | `mu`, `phi`, `Sigma` |
| $(\beta, \beta^*)$ | loadings excess returns на shocks | `beta`, `beta_star` |
| $(\lambda_0, \lambda_1)$ | prices of risk | `lambda0`, `lambda1` |
| $(\delta_0, \delta_1)$ | short-rate equation | `delta0`, `delta1` |
| $(A_n, B_n)$ | affine coefficients | `A`, `B` |
| $y_t^{(n), RN}$ | risk-neutral yield | `rny` |
| $TP_t^{(n)}$ | term premium | `tp` |

Ниже идем ровно в том порядке, в котором код выполняется в `__init__`.


## Шаг 0. Оркестровка в `__init__`

Это главный pipeline класса.

Смысл по шагам:
1. принять входную кривую и параметры;
2. агрегировать кривую к месячной частоте;
3. построить PCA-факторы;
4. собрать short-rate proxy;
5. посчитать excess returns;
6. оценить VAR, затем excess-return regression;
7. восстановить $(\lambda_0, \lambda_1)$;
8. оценить short-rate equation;
9. прогнать affine recursion;
10. получить fitted yield, risk-neutral yield, term premium и expected returns.


In [2]:
show_acm_lines(163, 222, title="`__init__`: порядок вычислений")

**`__init__`: порядок вычислений**

```python
 163:         self.n_factors = n_factors
 164:         self.curve = curve
 165: 
 166:         if selected_maturities is None:
 167:             self.selected_maturities = curve.columns
 168:         else:
 169:             self.selected_maturities = selected_maturities
 170: 
 171:         if curve_m is None:
 172:             self.curve_monthly = curve.resample('ME').mean() # used mean, but exist variant with date-to-date that may perform robuster, cause more extimating shocks
 173:         else:
 174:             self.curve_monthly = curve_m
 175: 
 176:         self.t_d = self.curve.shape[0]
 177:         self.t_m = self.curve_monthly.shape[0] - 1
 178:         self.n = self.curve.shape[1]
 179:         self.pc_factors_m, self.pc_factors_d, self.pc_loadings_m, self.pc_explained_m = self._get_pcs(self.curve_monthly, self.curve)
 180:         self.short_rate_proxy = self._prepare_short_rate_proxy(short_rate_proxy)
 181: 
 182:         self.rx_m = self._get_excess_returns()
 183: 
 184:         # ===== ACM Three-Step Regression =====
 185:         # 1st Step - Factor VAR
 186:         self.mu, self.phi, self.Sigma, self.v, self.s0 = self._estimate_var()
 187: 
 188:         # 2nd Step - Excess Returns
 189:         self.beta, self.omega, self.beta_star = self._excess_return_regression()
 190: 
 191:         # 3rd Step - Convexity-adjusted price of risk
 192:         self.lambda0, self.lambda1, self.mu_star, self.phi_star = self._retrieve_lambda()
 193: 
 194:         # Short Rate Equation
 195:         self.delta0, self.delta1 = self._short_rate_equation(
 196:             r1=self.short_rate_proxy,
 197:             X=self.pc_factors_m,
 198:         )
 199: 
 200:         # Affine Yield Coefficients
 201:         self.A, self.B = self._affine_coefficients(
 202:             lambda0=self.lambda0,
 203:             lambda1=self.lambda1,
 204:         )
 205: 
 206:         # Risk-Neutral Coefficients
 207:         self.Arn, self.Brn = self._affine_coefficients(
 208:             lambda0=np.zeros(self.lambda0.shape),
 209:             lambda1=np.zeros(self.lambda1.shape),
 210:         )
 211: 
 212:         # Model Implied Yield
 213:         self.miy = self._compute_yields(self.A, self.B)
 214: 
 215:         # Risk Neutral Yield
 216:         self.rny = self._compute_yields(self.Arn, self.Brn)
 217: 
 218:         # Term Premium
 219:         self.tp = self.miy - self.rny
 220: 
 221:         # Expected Return
 222:         self.er_loadings, self.er_hist = self._expected_return()
```

## Шаг 1. Проверки входа, месячная агрегация и короткая ставка

### 1.1 Проверки структуры кривой

Код требует:
- колонки кривой должны быть последовательными целыми числами `1, 2, ..., n`;
- `selected_maturities` должны существовать в кривой;
- если `curve_m` задан явно, он должен иметь те же колонки и месячную частоту.

### 1.2 Переход от daily к monthly

Если `curve_m is None`, то месячная кривая строится как среднее внутри месяца:

$$
\bar y_m^{(n)} = \frac{1}{D_m} \sum_{d \in m} y_d^{(n)}
$$

В коде это строка `curve.resample('ME').mean()`.

### 1.3 Short-rate proxy

Если прокси не передана, код берет первую колонку месячной кривой и делит на 12:

$$
r_t^{(1)} = y_t^{(1)} / 12
$$

Если прокси передана явно, она:
- сортируется;
- при необходимости ресемплится на месяц;
- выравнивается по индексу `curve_monthly`.


In [3]:
show_acm_lines(261, 328, title="`_assertions` + `_prepare_short_rate_proxy`")

**`_assertions` + `_prepare_short_rate_proxy`**

```python
 261:     @staticmethod
 262:     def _assertions(curve, curve_m, selected_maturities):
 263:         # Selected maturities are available
 264:         if selected_maturities is not None:
 265:             assert all([col in curve.columns for col in selected_maturities]), \
 266:                 "not all `selected_columns` are available in `curve`"
 267: 
 268:         # Consecutive monthly maturities
 269:         cond1 = curve.columns[0] != 1
 270:         cond2 = not all(np.diff(curve.columns.values) == 1)
 271:         if cond1 or cond2:
 272:             msg = "`curve` columns must be consecutive integers starting from 1"
 273:             raise AssertionError(msg)
 274: 
 275:         # Only if `curve_m` is passed
 276:         if curve_m is not None:
 277: 
 278:             # Same columns
 279:             assert curve_m.columns.equals(curve.columns), \
 280:                 "columns of `curve` and `curve_m` must be the same"
 281: 
 282:             # Monthly frequency
 283:             assert pd.infer_freq(curve_m.index) in {'M', 'ME'}, \
 284:                 "`curve_m` must have a DatetimeIndex with monthly frequency"
 285: 
 286:     def _prepare_short_rate_proxy(self, short_rate_proxy):
 287:         if short_rate_proxy is None:
 288:             proxy = (self.curve_monthly.iloc[:, 0] / 12).copy()
 289:             proxy.name = self.curve_monthly.columns[0]
 290:             return proxy
 291: 
 292:         if isinstance(short_rate_proxy, pd.Series):
 293:             proxy = short_rate_proxy.copy()
 294:         elif isinstance(short_rate_proxy, pd.DataFrame):
 295:             if short_rate_proxy.shape[1] != 1:
 296:                 raise ValueError(
 297:                     "`short_rate_proxy` DataFrame must have exactly one column"
 298:                 )
 299:             proxy = short_rate_proxy.iloc[:, 0].copy()
 300:         else:
 301:             raise TypeError(
 302:                 "`short_rate_proxy` must be a pandas Series or DataFrame"
 303:             )
 304: 
 305:         if not isinstance(proxy.index, pd.DatetimeIndex):
 306:             raise TypeError(
 307:                 "`short_rate_proxy` must have a DatetimeIndex"
 308:             )
 309: 
 310:         proxy = proxy.sort_index()
 311:         if proxy.index.has_duplicates:
 312:             proxy = proxy.groupby(level=0).mean()
 313: 
 314:         if not proxy.index.equals(self.curve_monthly.index):
 315:             proxy = proxy.resample('ME').mean()
 316: 
 317:         proxy = proxy.reindex(self.curve_monthly.index)
 318:         proxy = pd.to_numeric(proxy, errors='coerce')
 319: 
 320:         if proxy.isna().any():
 321:             missing_dates = proxy.index[proxy.isna()].strftime('%Y-%m-%d').tolist()
 322:             missing_dates = missing_dates[:5]
 323:             raise ValueError(
 324:                 "`short_rate_proxy` does not cover the monthly estimation "
 325:                 f"sample. Missing dates include: {missing_dates}"
 326:             )
 327: 
 328:         return proxy.rename(proxy.name or "short_rate_proxy")
```

## Шаг 2. Excess returns

Сначала код строит лог-цены:

$$
p_t^{(n)} = - \frac{n}{12} y_t^{(n)}
$$

Затем вычисляет excess return на облигации со сроком $n$ как доходность от удержания ее один период минус короткая ставка:

$$
rx_{t+1}^{(n)} = p_{t+1}^{(n-1)} - p_t^{(n)} - r_t^{(1)}
$$

Соответствие переменных:
- `ttm = np.arange(1, self.n + 1) / 12` это множитель $n/12$;
- `log_prices = - self.curve_monthly * ttm` это $p_t^{(n)}$;
- `shift(-1, axis=1)` сдвигает срок с $n$ на $n-1$;
- `rf = self.short_rate_proxy.shift(1)` это лаг короткой ставки.

Отдельно код принудительно ставит `rx[1] = 0`, потому что для 1M excess return не используется как обычная доходность более длинного бонда.


In [4]:
show_acm_lines(330, 339, title="`_get_excess_returns`")

**`_get_excess_returns`**

```python
 330:     def _get_excess_returns(self):
 331:         ttm = np.arange(1, self.n + 1) / 12
 332:         log_prices = - self.curve_monthly * ttm
 333:         rf = self.short_rate_proxy.shift(1)
 334:         rx = (log_prices - log_prices.shift(1, axis=0).shift(-1, axis=1)).subtract(rf, axis=0)
 335:         rx = rx.shift(1, axis=1)
 336: 
 337:         rx = rx.dropna(how='all', axis=0)
 338:         rx[1] = 0
 339:         return rx
```

## Шаг 3. PCA на кривой доходностей

Код сначала отбрасывает первые две зрелости:

$$
Y_t^{cut} = \big(y_t^{(3)}, y_t^{(4)}, \dots, y_t^{(n)}\big)
$$

Дальше центрирует месячную кривую по среднему и строит PCA:

$$
X_t = (Y_t^{cut} - \bar Y) W
$$

где:
- $W$ это `df_loadings`;
- $X_t$ это `pc_factors_m`.

Затем те же loadings применяются к daily-кривой:

$$
X_t^{daily} = (Y_t^{daily, cut} - \bar Y) W
$$

Именно поэтому параметры модели оцениваются по monthly-факторам, а fitted/risk-neutral yields потом можно получать и на daily частоте.


In [5]:
show_acm_lines(341, 381, title="`_get_pcs`")

**`_get_pcs`**

```python
 341:     def _get_pcs(self, curve_m, curve_d):
 342: 
 343:         # The authors' code shows that they ignore the first 2 maturities for
 344:         # the PC estimation.
 345:         curve_m_cut = curve_m.iloc[:, 2:]
 346:         curve_d_cut = curve_d.iloc[:, 2:]
 347: 
 348:         mean_yields = curve_m_cut.mean()
 349:         curve_m_cut = curve_m_cut - mean_yields
 350:         curve_d_cut = curve_d_cut - mean_yields
 351: 
 352:         pca = PCA(n_components=self.n_factors)
 353:         pca.fit(curve_m_cut)
 354:         col_names = [f'PC {i + 1}' for i in range(self.n_factors)]
 355:         df_loadings = pd.DataFrame(
 356:             data=pca.components_.T,
 357:             columns=col_names,
 358:             index=curve_m_cut.columns,
 359:         )
 360: 
 361:         df_pc_m = curve_m_cut @ df_loadings
 362:         sigma_factor = df_pc_m.std()
 363:         df_pc_m = df_pc_m / df_pc_m.std()
 364:         df_loadings = df_loadings / sigma_factor
 365: 
 366:         # Enforce average positive loadings
 367:         sign_changes = np.sign(df_loadings.mean())
 368:         df_loadings = sign_changes * df_loadings
 369:         df_pc_m = sign_changes * df_pc_m
 370: 
 371:         # Daily frequency
 372:         df_pc_d = curve_d_cut @ df_loadings
 373: 
 374:         # Percent Explained
 375:         df_explained = pd.Series(
 376:             data=pca.explained_variance_ratio_,
 377:             name='Explained Variance',
 378:             index=col_names,
 379:         )
 380: 
 381:         return df_pc_m, df_pc_d, df_loadings, df_explained
```

## Шаг 4. VAR(1) для факторного состояния

Факторная динамика задается как

$$
X_{t+1} = \mu + \Phi X_t + v_{t+1}, \qquad v_{t+1} \sim (0, \Sigma)
$$

В коде:
- `X_lhs` это $X_{t+1}$;
- `X_rhs` это константа и $X_t$;
- `var_coeffs = X_lhs @ pinv(X_rhs)` это OLS-оценка матрицы коэффициентов.

Важно: в этой реализации константа затем принудительно обнуляется:

$$
\mu = 0
$$

Это не просто математическая деталь, а конкретный выбор реализации.


In [6]:
show_acm_lines(383, 405, title="`_estimate_var`")

**`_estimate_var`**

```python
 383:     def _estimate_var(self):
 384:         X = self.pc_factors_m.copy().T
 385:         X_lhs = X.values[:, 1:]  # X_t+1. Left hand side of VAR
 386:         X_rhs = np.vstack((np.ones((1, self.t_m)), X.values[:, 0:-1]))  # X_t and a constant.
 387: 
 388:         var_coeffs = (X_lhs @ np.linalg.pinv(X_rhs))
 389: 
 390:         phi = var_coeffs[:, 1:]
 391: 
 392:         # Leave the estimated constant
 393:         # mu = var_coeffs[:, [0]]
 394: 
 395:         # Force constant to zero
 396:         mu = np.zeros((self.n_factors, 1))
 397:         var_coeffs[:, [0]] = 0
 398: 
 399:         # Residuals
 400:         v = X_lhs - var_coeffs @ X_rhs
 401:         Sigma = v @ v.T / (self.t_m - 1)
 402: 
 403:         s0 = np.cov(v).reshape((-1, 1))
 404: 
 405:         return mu, phi, Sigma, v, s0
```

## Шаг 5. Регрессия excess returns

На втором шаге ACM используется регрессия вида

$$
rx_{t+1}^{(n)} = a_n + c_n' X_t + \beta_n' v_{t+1} + e_{t+1}^{(n)}
$$

В коде матрица регрессоров `Z` содержит:
- константу;
- лагированное состояние `X`;
- инновации VAR `self.v`.

После оценки:
- `beta` это коэффициенты на `v_{t+1}`;
- `omega` это ковариация ошибок $e_{t+1}^{(n)}$;
- `beta_star[i, :] = kron(beta[i, :], beta[i, :])` нужен позже для convexity adjustment.


In [7]:
show_acm_lines(407, 428, title="`_excess_return_regression`")

**`_excess_return_regression`**

```python
 407:     def _excess_return_regression(self):
 408: 
 409:         if self.selected_maturities is not None:
 410:             rx = self.rx_m[self.selected_maturities].values
 411:         else:
 412:             rx = self.rx_m.values
 413: 
 414:         X = self.pc_factors_m.copy().T.values[:, :-1]
 415:         Z = np.vstack((np.ones((1, self.t_m)), X, self.v)).T  # Lagged X and Innovations
 416:         abc = inv(Z.T @ Z) @ (Z.T @ rx)
 417:         E = rx - Z @ abc
 418:         omega = np.var(E.reshape(-1, 1)) * np.eye(len(self.selected_maturities))
 419: 
 420:         abc = abc.T
 421:         beta = abc[:, -self.n_factors:]
 422: 
 423:         beta_star = np.zeros((len(self.selected_maturities), self.n_factors**2))
 424: 
 425:         for i in range(len(self.selected_maturities)):
 426:             beta_star[i, :] = np.kron(beta[i, :], beta[i, :]).T
 427: 
 428:         return beta, omega, beta_star
```

## Шаг 6. Восстановление prices of risk

Это центральный шаг ACM.

Сначала берутся те же excess returns, что и на шаге 5, и добавляется convexity adjustment:

$$
rx_{t+1}^{(n)} + \frac{1}{2}\big(\beta_n^* s_0 + \omega_n\big)
$$

В коде это строка

$$
\texttt{adjustment = beta\_star @ s0 + diag(omega)}
$$

Дальше решается линейная система на $(\lambda_0, \lambda_1)$:

$$
\mathbb E_t[rx_{t+1}^{(n)}] = \beta_n' \lambda_0 + \beta_n' \lambda_1 X_t - \frac{1}{2}\text{adj}_n
$$

В коде:
- `Y` это already adjusted expected returns;
- `X = self.beta` это матрица экспозиций на shocks;
- `Lambda = inv(X.T @ X) @ X.T @ Y` дает `lambda0` и `lambda1`.

После этого код сразу строит risk-neutral динамику факторов:

$$
\mu^* = \mu - \lambda_0, \qquad \Phi^* = \Phi - \lambda_1
$$


In [8]:
show_acm_lines(430, 451, title="`_retrieve_lambda`")

**`_retrieve_lambda`**

```python
 430:     def _retrieve_lambda(self):
 431:         rx = self.rx_m[self.selected_maturities]
 432:         factors = np.hstack([np.ones((self.t_m, 1)), self.pc_factors_m.iloc[:-1].values])
 433: 
 434:         # Orthogonalize factors with respect to v
 435:         v_proj = self.v.T @ np.linalg.pinv(self.v @ self.v.T) @ self.v
 436:         factors = factors - v_proj @ factors
 437: 
 438:         adjustment = self.beta_star @ self.s0 + np.diag(self.omega).reshape(-1, 1)
 439:         rx_adjusted = rx.values + (1 / 2) * np.tile(adjustment, (1, self.t_m)).T
 440:         Y = (inv(factors.T @ factors) @ factors.T @ rx_adjusted).T
 441: 
 442:         # Compute Lambda
 443:         X = self.beta
 444:         Lambda = inv(X.T @ X) @ X.T @ Y
 445:         lambda0 = Lambda[:, 0]
 446:         lambda1 = Lambda[:, 1:]
 447: 
 448:         muStar = self.mu.reshape(-1) - lambda0
 449:         phiStar = self.phi - lambda1
 450: 
 451:         return lambda0, lambda1, muStar, phiStar
```

## Шаг 7. Short-rate equation

Короткая ставка аппроксимируется линейным уравнением по факторам:

$$
r_t^{(1)} = \delta_0 + \delta_1' X_t
$$

В коде:
- `r1` это уже подготовленная `short_rate_proxy`;
- `add_constant(X)` добавляет свободный член;
- `Delta = inv(X.T @ X) @ X.T @ r1` это OLS-оценка.


In [9]:
show_acm_lines(453, 460, title="`_short_rate_equation`")

**`_short_rate_equation`**

```python
 453:     @staticmethod
 454:     def _short_rate_equation(r1, X):
 455:         r1 = pd.Series(r1).reindex(X.index)
 456:         X = add_constant(X)
 457:         Delta = inv(X.T @ X) @ X.T @ r1
 458:         delta0 = Delta.iloc[0]
 459:         delta1 = Delta.iloc[1:].values
 460:         return delta0, delta1
```

## Шаг 8. Affine recursion

После получения $(\delta_0, \delta_1)$ и $(\lambda_0, \lambda_1)$ код строит affine coefficients рекурсией.

Для первого срока:

$$
A_1 = -\delta_0, \qquad B_1 = -\delta_1
$$

Для следующих сроков:

$$
A_{n+1} = A_n + B_n'(\mu - \lambda_0) + \frac{1}{2}(B_n' \Sigma B_n + \omega) + A_1
$$

$$
B_{n+1} = B_n'(\Phi - \lambda_1) + B_1
$$

В коде:
- `Bpb = np.kron(B[n - 1, :], B[n - 1, :])` это способ записать квадратичный член;
- `s0term` это convexity part;
- один и тот же метод вызывается и для physical measure, и для risk-neutral measure.


In [10]:
show_acm_lines(462, 478, title="`_affine_coefficients`")

**`_affine_coefficients`**

```python
 462:     def _affine_coefficients(self, lambda0, lambda1):
 463:         lambda0 = lambda0.reshape(-1, 1)
 464: 
 465:         A = np.zeros(self.n)
 466:         B = np.zeros((self.n, self.n_factors))
 467: 
 468:         A[0] = - self.delta0
 469:         B[0, :] = - self.delta1
 470: 
 471:         for n in range(1, self.n):
 472:             Bpb = np.kron(B[n - 1, :], B[n - 1, :])
 473:             s0term = 0.5 * (Bpb @ self.s0 + self.omega[0, 0])
 474: 
 475:             A[n] = (A[n - 1] + B[n - 1, :] @ (self.mu - lambda0) + s0term + A[0])[0]
 476:             B[n, :] = B[n - 1, :] @ (self.phi - lambda1) + B[0, :]
 477: 
 478:         return A, B
```

## Шаг 9. Доходности, risk-neutral yield и term premium

После рекурсии доходности считаются как

$$
y_t^{(n)} = -\frac{A_n + B_n' X_t}{n/12}
$$

В коде это делает `_compute_yields`.

Дальше в `__init__` собираются два набора коэффициентов:
- `A, B` для fitted yields;
- `Arn, Brn` для risk-neutral yields.

Итоговые объекты:
- `miy` = fitted yield;
- `rny` = risk-neutral yield;
- `tp` = `miy - rny`.

Важно: именно в этой реализации term premium считается как разность fitted и risk-neutral доходностей. Если нужно сравнивать с определением через observed yield, это уже отдельная модификация поверх текущего кода.


In [11]:
show_acm_lines(200, 219, title="сборка `miy`, `rny`, `tp` в `__init__`")
show_acm_lines(480, 489, title="`_compute_yields`")

**сборка `miy`, `rny`, `tp` в `__init__`**

```python
 200:         # Affine Yield Coefficients
 201:         self.A, self.B = self._affine_coefficients(
 202:             lambda0=self.lambda0,
 203:             lambda1=self.lambda1,
 204:         )
 205: 
 206:         # Risk-Neutral Coefficients
 207:         self.Arn, self.Brn = self._affine_coefficients(
 208:             lambda0=np.zeros(self.lambda0.shape),
 209:             lambda1=np.zeros(self.lambda1.shape),
 210:         )
 211: 
 212:         # Model Implied Yield
 213:         self.miy = self._compute_yields(self.A, self.B)
 214: 
 215:         # Risk Neutral Yield
 216:         self.rny = self._compute_yields(self.Arn, self.Brn)
 217: 
 218:         # Term Premium
 219:         self.tp = self.miy - self.rny
```

**`_compute_yields`**

```python
 480:     def _compute_yields(self, A, B):
 481:         A = A.reshape(-1, 1)
 482:         multiplier = np.tile(self.curve.columns / 12, (self.t_d, 1)).T
 483:         yields = (- ((np.tile(A, (1, self.t_d)) + B @ self.pc_factors_d.T) / multiplier).T).values
 484:         yields = pd.DataFrame(
 485:             data=yields,
 486:             index=self.curve.index,
 487:             columns=self.curve.columns,
 488:         )
 489:         return yields
```

## Шаг 10. Expected returns

Последний блок строит два выхода.

### 10.1 Loadings expected returns on factors

$$
\text{ER loadings}_n = B_n \lambda_1 \cdot \text{sd}(X)
$$

Это `er_loadings`.

### 10.2 Историческая in-sample оценка expected return

$$
\widehat{ER}_t^{(n)} = B_n(\lambda_1 X_t + \lambda_0) - \frac{1}{2}\text{conv adj}_n
$$

Это `er_hist`.

То есть после affine-рекурсии код получает не только decomposition yield curve, но и implied expected excess returns.


In [12]:
show_acm_lines(491, 516, title="`_expected_return`")

**`_expected_return`**

```python
 491:     def _expected_return(self):
 492:         """
 493:         Compute the "expected return" and "convexity adjustment" terms, to get
 494:         the expected return loadings and historical estimate
 495: 
 496:         Loadings are interpreted as the effect of 1sd of the PCs on the
 497:         expected returns
 498:         """
 499:         stds = self.pc_factors_m.std().values[:, None].T
 500:         er_loadings = (self.B @ self.lambda1) * stds
 501:         er_loadings = pd.DataFrame(
 502:             data=er_loadings,
 503:             columns=self.pc_factors_m.columns,
 504:             index=range(1, self.n + 1),
 505:         )
 506: 
 507:         # Historical estimate
 508:         exp_ret = (self.B @ (self.lambda1 @ self.pc_factors_d.T + self.lambda0.reshape(-1, 1))).values
 509:         conv_adj = np.diag(self.B @ self.Sigma @ self.B.T) + self.omega[0, 0]
 510:         er_hist = (exp_ret - 0.5 * conv_adj[:, None]).T
 511:         er_hist_d = pd.DataFrame(
 512:             data=er_hist,
 513:             index=self.pc_factors_d.index,
 514:             columns=self.curve.columns,
 515:         )
 516:         return er_loadings, er_hist_d
```

## Что в этой реализации особенно важно проверять

Если результаты расходятся с другой реализацией или с заметкой ЦБ, то в первую очередь смотреть сюда:

1. `curve.resample('ME').mean()`
   Это определяет месячную агрегацию.

2. `_get_pcs`: `curve_m.iloc[:, 2:]`
   Первые две зрелости исключаются из PCA.

3. `_estimate_var`: `mu = np.zeros(...)`
   Константа VAR принудительно обнуляется.

4. `_prepare_short_rate_proxy`
   Здесь определяется, что именно модель считает короткой ставкой.

5. `_excess_return_regression`: `omega = scalar * I`
   Ошибки второй регрессии диагональные и одинакового масштаба.

6. `self.tp = self.miy - self.rny`
   Это `tp_fit`, а не `observed - risk-neutral`.

Этот список и есть минимальный чек-лист для сопоставления формул статьи с текущим кодом.
